In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('Loan_Default.csv')
df=df.drop(columns=['LoanID'])

In [3]:
df.sample(5)

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
40694,57,138276,65131,370,76,1,13.75,48,0.30,Master's,Self-employed,Divorced,Yes,Yes,Other,No,0
143966,50,101342,205002,407,60,2,15.60,36,0.88,PhD,Part-time,Married,Yes,No,Auto,No,0
255044,28,132201,24323,801,68,4,3.72,48,0.43,Bachelor's,Part-time,Married,Yes,Yes,Home,No,0
21837,42,19253,242930,754,84,4,16.56,24,0.18,Master's,Unemployed,Single,Yes,No,Education,Yes,0
76088,36,122754,145940,768,57,4,3.81,60,0.68,Master's,Unemployed,Divorced,Yes,No,Auto,No,0


In [4]:
df.shape

(255347, 17)

In [5]:
df.isnull().mean()*100

Age               0.0
Income            0.0
LoanAmount        0.0
CreditScore       0.0
MonthsEmployed    0.0
NumCreditLines    0.0
InterestRate      0.0
LoanTerm          0.0
DTIRatio          0.0
Education         0.0
EmploymentType    0.0
MaritalStatus     0.0
HasMortgage       0.0
HasDependents     0.0
LoanPurpose       0.0
HasCoSigner       0.0
Default           0.0
dtype: float64

In [6]:
# there are no empty / null rows

In [7]:
df['NumCreditLines'].value_counts()

df['Education'] = df['Education'].replace({
    "Bachelor's": "Bachelors"
})
df['Education'] = df['Education'].replace({
    "Master's": "Masters"
})
df['Education'].value_counts()

Education
Bachelors      64366
High School    63903
Masters        63541
PhD            63537
Name: count, dtype: int64

In [8]:
df['LoanTerm'].value_counts()

LoanTerm
48    51166
60    51154
36    51061
24    51009
12    50957
Name: count, dtype: int64

In [9]:
df['LoanPurpose'].value_counts()

LoanPurpose
Business     51298
Home         51286
Education    51005
Other        50914
Auto         50844
Name: count, dtype: int64

In [10]:
num_cols=['Age','Income','LoanAmount','CreditScore','MonthsEmployed','NumCreditLines','InterestRate','LoanTerm','DTIRatio']
ord_cols=['Education']
cat_cols=['EmploymentType','MaritalStatus','HasMortgage','HasDependents','LoanPurpose','HasCoSigner']

In [11]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [12]:
num_pipeline=Pipeline([
    ('imputer',SimpleImputer()),
    ('Scalar',StandardScaler())
])

In [13]:
ord_pipeline=Pipeline([
    ('ordinal',OrdinalEncoder(categories=[['High School','Bachelors','Masters','PhD']]))
])

In [14]:
cat_pipeline=Pipeline([
    ('categorical',OneHotEncoder(sparse_output=False,dtype=int,drop='first'))
])

In [15]:
transformer=ColumnTransformer(
    transformers=[
        ('tnf1',num_pipeline,num_cols),
        ('tnf2',ord_pipeline,ord_cols),
        ('tnf3',cat_pipeline,cat_cols),
    ]
)

In [16]:
pipe=Pipeline([
    ('preprocessing',transformer),
    ('model',LogisticRegression(class_weight='balanced'))
])

In [17]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['Default']),df['Default'],test_size=0.25)

In [18]:
pipe.fit(x_train,y_train)
# print(x_train.columns)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('tnf1', ...), ('tnf2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
pred=pipe.predict(x_test)

In [20]:
acc=accuracy_score(y_test,pred)

In [21]:
acc

0.6757209768629479

In [22]:
from sklearn.metrics import classification_report
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.94      0.68      0.79     56441
           1       0.22      0.68      0.33      7396

    accuracy                           0.68     63837
   macro avg       0.58      0.68      0.56     63837
weighted avg       0.86      0.68      0.73     63837

